This notebook compiles various tools used throughout the project, such as splitting a folder into subfolders or normalizing file names. These utilities are not essential but can be helpful.

In [ ]:
import os
import shutil

CURRENT_DATASET = "lapresse"
FOLDER_PATH = os.path.join("..", "data", "cleaned_datasets", f"dataset_{CURRENT_DATASET}")

In [ ]:


def split_folder(folder_path, num_chunks=3):
    # Get all files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    
    # Sort to ensure consistency
    files.sort()

    # Determine chunk size
    chunk_size = len(files) // num_chunks
    remainder = len(files) % num_chunks

    # Create subfolders and distribute files
    start_idx = 0
    for i in range(num_chunks):
        chunk_folder = os.path.join(folder_path, f"folder-{i+1}")
        os.makedirs(chunk_folder, exist_ok=True)
        
        end_idx = start_idx + chunk_size + (1 if i < remainder else 0)  # Distribute remainder files
        for file in files[start_idx:end_idx]:
            shutil.move(os.path.join(folder_path, file), os.path.join(chunk_folder, file))

        start_idx = end_idx

# Example usage
split_folder(FOLDER_PATH)


In [ ]:
main_folder = FOLDER_PATH

# Iterate through subfolders
for subfolder in os.listdir(main_folder):
    subfolder_path = os.path.join(main_folder, subfolder)
    
    if os.path.isdir(subfolder_path):
        # Move all files from subfolder to main folder
        for file in os.listdir(subfolder_path):
            src_path = os.path.join(subfolder_path, file)
            dest_path = os.path.join(main_folder, file)
            shutil.move(src_path, dest_path)

        # Remove the now empty subfolder
        os.rmdir(subfolder_path)

In [ ]:
import os, re
import unicodedata
def normalize_symbols(text):
    text = re.sub(r"[‐‑‒–—―]", "-", text)  # Normalize all dashes to `-`
    text = text.replace("＄", "$")  # Normalize fullwidth dollar
    return text

def normalize_filenames_in_directory(directory):
    """
    Normalize all filenames in 'directory' to NFC form.

    If a filename is already in NFC, it will remain unchanged.
    If normalizing causes a collision (two different files normalize to the same name),
    a warning is printed, and the rename is skipped.
    """
    print(len(os.listdir(directory)), "file normalized")
    for fname in os.listdir(directory):
        old_path = os.path.join(directory, fname)
        if not os.path.isfile(old_path):
            # Skip subdirectories or non-regular files if necessary
            continue

        # Normalize the filename to NFC form
        norm_fname = normalize_symbols(unicodedata.normalize('NFC', fname)).strip()
        new_path = os.path.join(directory, norm_fname)

        # Only rename if the normalized name is different
        if norm_fname != fname:
            if os.path.exists(new_path):
                print(f"Collision: '{old_path}' cannot be renamed to '{new_path}' because it already exists.")
            else:
                os.rename(old_path, new_path)
                print(f"Renamed:\n  {old_path}\n  -> {new_path}")

# Example usage:
PATH = os.path.join("..", "..", "baseline_mistral")
normalize_filenames_in_directory(PATH)


In [ ]:
import os

path = os.path.join("..", "data", "LORA-tuning_Llama-3.2-3B_summaries")

for fname in os.listdir(path):
    file_path = os.path.join(path, fname)
    text = open(file_path, 'r').read()
    text = text.split("<résumé>: ")[-1].strip()
    if len(text) == 0: text = "Aucun résumé..."
    open(file_path, 'w').write(text)

In [ ]:
out_of_bound_mask = (df["llm_score"] < 0) | (df["llm_score"] > 10)
out_of_bound_filenames = df.loc[out_of_bound_mask, "filename"].tolist()
print("Out-of-bound filenames:", out_of_bound_filenames)

# Create a lookup dictionary from your original data.
# 'data' should be a list of tuples: (filename, initial_text, summary)
data_lookup = {entry[0]: (entry[1], entry[2]) for entry in data}

# Recompute llm_score for each out-of-bound row
for idx in tqdm(df.index[out_of_bound_mask], desc="Recalculating llm_score"):
    filename = df.at[idx, "filename"]
    if filename in data_lookup:
        initial_text, summary = data_lookup[filename]
        new_llm_score = score_summary_llm(model, tokenizer, initial_text, summary)
        df.at[idx, "llm_score"] = new_llm_score

# Save the updated metrics back to disk
np.save(METRICS_FILE_PATH, df.to_numpy())

print("Updated metrics saved.")

In [ ]:
import os
import re
import unicodedata
import hashlib
import pandas as pd

# Paths
DATASET_PATH = "../data/cleaned_datasets/dataset_lapresse"
BACKUP_MAPPING_PATH = "filename_mapping.csv"

def normalize_filename(filename, max_length=50):
    """Normalize filename: lowercase, remove accents, replace spaces, add hash for uniqueness."""
    base, ext = os.path.splitext(filename)
    normalized = unicodedata.normalize('NFKD', base).encode('ascii', 'ignore').decode('ascii')
    normalized = normalized.lower()
    normalized = re.sub(r'\s+', '_', normalized)
    normalized = re.sub(r'[^\w-]', '', normalized)  # Keep only alphanumerics, _, -
    normalized = re.sub(r'_+', '_', normalized)  # Remove multiple underscores

    if len(normalized) > max_length:
        normalized = normalized[:max_length].rstrip('_')

    hash_suffix = hashlib.sha1(filename.encode('utf-8')).hexdigest()[:8]
    return f"{normalized}_{hash_suffix}{ext.lower()}"

# Get all filenames
filenames = os.listdir(DATASET_PATH)

# Store original → new mappings
mapping = []
renamed_files = set()

for filename in filenames:
    new_filename = normalize_filename(filename)

    # Ensure uniqueness in case two different files get the same name
    counter = 1
    base, ext = os.path.splitext(new_filename)
    while new_filename in renamed_files:
        new_filename = f"{base}_{counter}{ext}"
        counter += 1

    renamed_files.add(new_filename)

    # Rename the file
    old_path = os.path.join(DATASET_PATH, filename)
    new_path = os.path.join(DATASET_PATH, new_filename)
    
    try:
        os.rename(old_path, new_path)
        mapping.append((filename, new_filename))
    except Exception as e:
        print(f"Error renaming {filename}: {e}")

# Save mapping for reference
df = pd.DataFrame(mapping, columns=["original_filename", "new_filename"])
df.to_csv(BACKUP_MAPPING_PATH, index=False, encoding="utf-8")
print(f"Renamed {len(mapping)} files. Backup saved to {BACKUP_MAPPING_PATH}.")


In [ ]:
import os
import pandas as pd

# Directory containing the original files
DATASET_PATH = "../data/cleaned_datasets/dataset_lapresse"

# List all filenames in the directory
filenames = os.listdir(DATASET_PATH)

# Save the original filenames to a CSV file
df = pd.DataFrame({"original_filename": filenames})
df.to_csv("original_filenames.csv", index=False, encoding="utf-8")

print(f"Saved {len(filenames)} filenames to original_filenames.csv")


In [ ]:
import os
import pandas as pd

def encode_filenames(dataset_path, mapping_path, strict=True):
    """
    Rename files in the dataset directory using a mapping.
    
    For each file in dataset_path:
      - If the file is present in the mapping (original_filename → new_filename), rename it.
      - If not and strict=True, raise an error.
      - If not and strict=False, skip the file.
    
    Args:
        dataset_path (str): Directory containing the files to be renamed.
        mapping_path (str): Path to CSV file with two columns: "original_filename" and "new_filename".
        strict (bool): If True, raise an error for any file not found in the mapping.
    """
    # Check if the mapping file exists
    if not os.path.exists(mapping_path):
        raise FileNotFoundError(f"Mapping file '{mapping_path}' not found.")

    # Load the mapping file
    df = pd.read_csv(mapping_path)
    mapping = dict(zip(df["original_filename"], df["new_filename"]))

    # Get list of all files in the dataset directory
    files = os.listdir(dataset_path)

    for filename in files:
        if filename == ".DS_Store":
            os.remove(os.path.join(dataset_path, filename))
            continue
        if filename in mapping:
            old_path = os.path.join(dataset_path, filename)
            new_path = os.path.join(dataset_path, mapping[filename])
            os.rename(old_path, new_path)
            print(f"Renamed '{filename}' to '{mapping[filename]}'")
        else:
            if filename in mapping.values(): continue
            if strict:
                raise ValueError(f"File '{filename}' not found in mapping.")
            else:
                print(f"Skipping '{filename}' because it's not in the mapping.")

# Example usage:
DATASET_PATH = os.path.join("..", "..", "baseline_generated_summaries", "baseline_mistral")
MAPPING_PATH = os.path.join("..", "..", "filename_mapping.csv")

encode_filenames(DATASET_PATH, MAPPING_PATH, strict=True)


In [ ]:
import os
import numpy as np
import pandas as pd

def update_dataset_filenames(dataset_path, mapping_path, output_path, strict=True):
    """
    Update the filename column in a dataset using a provided mapping.
    
    The dataset can be a CSV file or a NumPy file. The function assumes the dataset
    has a column named "filename" (singular) that contains the file names to update.
    
    Args:
        dataset_path (str): Path to the dataset (CSV or .npy file).
        mapping_path (str): Path to the CSV file with mapping columns:
                            "original_filename" and "new_filename".
        output_path (str): Path where the updated dataset will be saved.
        strict (bool): If True, raise an error when a filename is not found in the mapping.
                       If False, leaves unmapped filenames unchanged.
    
    Raises:
        ValueError: If the dataset format is unsupported or if a filename is missing in the mapping (in strict mode).
    """
    # Load dataset
    ext = os.path.splitext(dataset_path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(dataset_path)
    elif ext == ".npy":
        data = np.load(dataset_path, allow_pickle=True)
        # Assuming the numpy file contains an array of lists/dicts convertible to DataFrame.
        df = pd.DataFrame(data.tolist())
    else:
        raise ValueError("Unsupported dataset format. Use CSV or NumPy (.npy) file.")

    # Ensure the dataset has the "filename" column.
    if "filename" not in df.columns:
        raise ValueError("Dataset does not contain a 'filename' column.")

    # Load the mapping file.
    mapping_df = pd.read_csv(mapping_path)
    if "original_filename" not in mapping_df.columns or "new_filename" not in mapping_df.columns:
        raise ValueError("Mapping file must contain 'original_filename' and 'new_filename' columns.")
    
    # Build mapping: original_filename -> new_filename.
    mapping = dict(zip(mapping_df["original_filename"], mapping_df["new_filename"]))

    # Define a function to update a single filename.
    def update_filename(fn):
        if fn in mapping:
            return mapping[fn]
        else:
            if strict:
                raise ValueError(f"Filename '{fn}' not found in mapping!")
            else:
                return fn

    # Apply the mapping to the "filename" column.
    df["filename"] = df["filename"].apply(update_filename)

    # Save the updated dataset.
    if ext == ".csv":
        df.to_csv(output_path, index=False)
    elif ext == ".npy":
        np.save(output_path, df.to_numpy())

    print(f"Updated dataset saved to {output_path}.")

# Example usage:
DATASET_PATH = "../data/curriculum_metrics.csv"  # or .npy
MAPPING_PATH = "filename_mapping.csv"
OUTPUT_PATH = "curriculum_metrics2.csv"  # or .npy

update_dataset_filenames(DATASET_PATH, MAPPING_PATH, OUTPUT_PATH, strict=True)
